In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /scratch/jieq/pandax/dias_notebooks/gksriharsha/eda-speedtests/src/small_bench/checkpoints/post_cell_7.pickle

In [ ]:
%%cudf.pandas.profile
### cell 8 ###

cols = ['Avg. Avg D Kbps', 'Avg. Avg U Kbps', 'Avg Lat Ms']

# Mobile stats: group and agg first/last, then compute deltas without .xs
mobile_agg = Mobile_df.groupby('Name')[cols].agg(['first', 'last'])
Mobile_Stats = (
    (mobile_agg[('Avg. Avg D Kbps', 'last')] - mobile_agg[('Avg. Avg D Kbps', 'first')])
    .to_frame('Change_Download')
    .assign(
        Change_Upload=(mobile_agg[('Avg. Avg U Kbps', 'last')] - mobile_agg[('Avg. Avg U Kbps', 'first')]),
        Change_Latency=(mobile_agg[('Avg Lat Ms',       'last')] - mobile_agg[('Avg Lat Ms',       'first')])
    )
)

# Broadband stats: same pattern
broad_agg = Broadband_df.groupby('Name')[cols].agg(['first', 'last'])
Broadband_Stats = (
    (broad_agg[('Avg. Avg D Kbps', 'last')] - broad_agg[('Avg. Avg D Kbps', 'first')])
    .to_frame('Change_Download')
    .assign(
        Change_Upload=(broad_agg[('Avg. Avg U Kbps', 'last')] - broad_agg[('Avg. Avg U Kbps', 'first')]),
        Change_Latency=(broad_agg[('Avg Lat Ms',       'last')] - broad_agg[('Avg Lat Ms',       'first')])
    )
)

# Combine only the download-change columns
Total_Stats = (
    Mobile_Stats['Change_Download'].to_frame('Mobile')
    .join(Broadband_Stats['Change_Download'].to_frame('Broadband'))
)